# So sánh các phương pháp xử lý ảnh
Notebook này so sánh:
1. **Phát hiện biên**: Canny vs Sobel vs Laplacian
2. **Phát hiện đường thẳng**: Standard Hough Transform vs Probabilistic Hough Transform

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import os
from pathlib import Path

folder_path = Path('../data/raw/')
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
image_list = sorted([
    str(p) for p in folder_path.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
])
print(f'Tìm thấy {len(image_list)} ảnh: {[os.path.basename(p) for p in image_list]}')

## Hàm tiền xử lý dùng chung

In [ ]:
def preprocess(img_path, target_width=800, target_height=600, crop_margin=50):
    """Đọc ảnh, resize+crop, grayscale, CLAHE, Gaussian blur."""
    img = cv2.imread(img_path)
    img = cv2.resize(img, (target_width, target_height))
    img = img[crop_margin:target_height - crop_margin, crop_margin:target_width - crop_margin]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    return gray

def load_rgb(img_path, target_width=800, target_height=600, crop_margin=50):
    """Đọc ảnh gốc dạng RGB đã crop, dùng để hiển thị."""
    img = cv2.imread(img_path)
    img = cv2.resize(img, (target_width, target_height))
    img = img[crop_margin:target_height - crop_margin, crop_margin:target_width - crop_margin]
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

---
## Phần 1: So sánh phương pháp phát hiện biên

| Phương pháp | Nguyên lý | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **Canny** | Gradient + non-max suppression + hysteresis threshold | Biên mỏng, ít nhiễu, chuẩn | Cần tinh chỉnh 2 ngưỡng |
| **Sobel** | Đạo hàm bậc 1 theo x và y | Đơn giản, nhanh | Nhạy với nhiễu, biên dày hơn |
| **Laplacian** | Đạo hàm bậc 2 | Phát hiện được cả chi tiết nhỏ | Rất nhạy với nhiễu |

In [ ]:
def edge_canny(img_gray, low=100, high=250):
    return cv2.Canny(img_gray, low, high)

def edge_sobel(img_gray):
    """Tính magnitude gradient từ Sobel X và Y."""
    sx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sx**2 + sy**2)
    # Normalize về [0, 255] và threshold để rõ biên
    magnitude = np.clip(magnitude, 0, 255).astype(np.uint8)
    _, binary = cv2.threshold(magnitude, 50, 255, cv2.THRESH_BINARY)
    return binary

def edge_laplacian(img_gray):
    """Laplacian đạo hàm bậc 2, lấy giá trị tuyệt đối."""
    lap = cv2.Laplacian(img_gray, cv2.CV_64F, ksize=3)
    lap = np.clip(np.abs(lap), 0, 255).astype(np.uint8)
    _, binary = cv2.threshold(lap, 15, 255, cv2.THRESH_BINARY)
    return binary

In [ ]:
# So sánh trực quan trên từng ảnh
for img_path in image_list:
    name = os.path.basename(img_path)
    gray = preprocess(img_path)
    original = load_rgb(img_path)

    canny  = edge_canny(gray)
    sobel  = edge_sobel(gray)
    laplacian = edge_laplacian(gray)

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(f'So sánh phát hiện biên — {name}', fontsize=14, fontweight='bold')

    for ax, img, title in zip(axes,
        [original, canny, sobel, laplacian],
        ['Ảnh gốc', 'Canny', 'Sobel', 'Laplacian']):
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

## Đánh giá tốc độ (thời gian xử lý trung bình)

In [ ]:
N_RUNS = 20  # chạy N lần để tính trung bình

# Dùng ảnh đầu tiên để benchmark
gray_sample = preprocess(image_list[0])

methods = {
    'Canny': lambda g: edge_canny(g),
    'Sobel': lambda g: edge_sobel(g),
    'Laplacian': lambda g: edge_laplacian(g),
}

times = {}
for method_name, fn in methods.items():
    start = time.perf_counter()
    for _ in range(N_RUNS):
        fn(gray_sample)
    elapsed = (time.perf_counter() - start) / N_RUNS * 1000  # ms
    times[method_name] = elapsed
    print(f'{method_name:12s}: {elapsed:.2f} ms / ảnh')

# Biểu đồ cột
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(times.keys(), times.values(), color=['steelblue', 'tomato', 'seagreen'])
ax.set_ylabel('Thời gian (ms)')
ax.set_title(f'Thời gian xử lý trung bình ({N_RUNS} lần chạy)')
for bar, val in zip(bars, times.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val:.2f}ms', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## Đánh giá mật độ biên (Edge Density)
Tỉ lệ pixel biên / tổng pixel — phản ánh mức độ nhiễu và độ nhạy của từng phương pháp.

In [ ]:
densities = {'Canny': [], 'Sobel': [], 'Laplacian': []}

for img_path in image_list:
    gray = preprocess(img_path)
    total = gray.size
    densities['Canny'].append(np.count_nonzero(edge_canny(gray)) / total * 100)
    densities['Sobel'].append(np.count_nonzero(edge_sobel(gray)) / total * 100)
    densities['Laplacian'].append(np.count_nonzero(edge_laplacian(gray)) / total * 100)

img_names = [os.path.basename(p) for p in image_list]
x = np.arange(len(img_names))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 5))
for i, (method, vals) in enumerate(densities.items()):
    ax.bar(x + i * width, vals, width, label=method)

ax.set_xlabel('Ảnh')
ax.set_ylabel('Edge density (%)')
ax.set_title('Mật độ biên theo từng ảnh và từng phương pháp')
ax.set_xticks(x + width)
ax.set_xticklabels(img_names, rotation=30, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

print('\nMật độ biên trung bình:')
for method, vals in densities.items():
    print(f'  {method:12s}: {np.mean(vals):.2f}%')

---
## Phần 2: So sánh phương pháp phát hiện đường thẳng

| Phương pháp | Nguyên lý | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **Standard Hough** (`HoughLines`) | Vote tất cả điểm trong không gian (ρ, θ) | Tìm được đường thẳng dài, liên tục | Trả về đường vô hạn, phải tự vẽ đoạn thẳng |
| **Probabilistic Hough** (`HoughLinesP`) | Chỉ vote trên mẫu ngẫu nhiên | Trả về đoạn thẳng trực tiếp, nhanh hơn | Có thể bỏ sót đường ngắn |

In [ ]:
def region_of_interest(img):
    h, w = img.shape
    mask = np.zeros_like(img)
    poly = np.array([[(0, h), (w, h), (w, int(h * 0.3)), (0, int(h * 0.3))]])
    cv2.fillPoly(mask, poly, 255)
    return cv2.bitwise_and(img, mask)

def lines_standard_hough(edges, img_rgb):
    """Standard Hough Transform — trả về đường thẳng vô hạn."""
    out = img_rgb.copy()
    lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=120)
    if lines is not None:
        for rho, theta in lines[:, 0]:
            a, b = np.cos(theta), np.sin(theta)
            x0, y0 = a * rho, b * rho
            x1 = int(x0 + 1000 * (-b))
            y1 = int(y0 + 1000 * a)
            x2 = int(x0 - 1000 * (-b))
            y2 = int(y0 - 1000 * a)
            cv2.line(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
    n = len(lines) if lines is not None else 0
    return out, n

def lines_probabilistic_hough(edges, img_rgb):
    """Probabilistic Hough Transform — trả về đoạn thẳng."""
    out = img_rgb.copy()
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180,
                            threshold=150, minLineLength=220, maxLineGap=80)
    if lines is not None:
        for x1, y1, x2, y2 in lines[:, 0]:
            cv2.line(out, (x1, y1), (x2, y2), (255, 0, 0), 2)
    n = len(lines) if lines is not None else 0
    return out, n

In [ ]:
for img_path in image_list:
    name = os.path.basename(img_path)
    gray = preprocess(img_path)
    original = load_rgb(img_path)

    edges = cv2.Canny(gray, 100, 250)
    roi_edges = region_of_interest(edges)

    std_result, n_std   = lines_standard_hough(roi_edges, original)
    prob_result, n_prob = lines_probabilistic_hough(roi_edges, original)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    fig.suptitle(f'So sánh Hough Transform — {name}', fontsize=14, fontweight='bold')

    axes[0].imshow(original)
    axes[0].set_title('Ảnh gốc', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(std_result)
    axes[1].set_title(f'Standard HoughLines\n({n_std} đường, màu xanh lá)', fontsize=12)
    axes[1].axis('off')

    axes[2].imshow(prob_result)
    axes[2].set_title(f'Probabilistic HoughLinesP\n({n_prob} đoạn, màu đỏ)', fontsize=12)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

## Đánh giá tốc độ — Hough Transform

In [ ]:
gray_sample = preprocess(image_list[0])
edges_sample = region_of_interest(cv2.Canny(gray_sample, 100, 250))
rgb_sample   = load_rgb(image_list[0])

line_methods = {
    'Standard Hough': lambda: lines_standard_hough(edges_sample, rgb_sample),
    'Probabilistic Hough': lambda: lines_probabilistic_hough(edges_sample, rgb_sample),
}

line_times = {}
for method_name, fn in line_methods.items():
    start = time.perf_counter()
    for _ in range(N_RUNS):
        fn()
    elapsed = (time.perf_counter() - start) / N_RUNS * 1000
    line_times[method_name] = elapsed
    print(f'{method_name}: {elapsed:.2f} ms / ảnh')

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(line_times.keys(), line_times.values(), color=['seagreen', 'steelblue'])
ax.set_ylabel('Thời gian (ms)')
ax.set_title(f'Thời gian xử lý trung bình — Hough ({N_RUNS} lần)')
for bar, val in zip(bars, line_times.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{val:.2f}ms', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

---
## Tổng kết

### Phát hiện biên
- **Canny**: Tốt nhất cho bài toán này — biên mỏng, sạch, ít nhiễu. Edge density thấp nhất nhưng chất lượng cao nhất.
- **Sobel**: Biên dày hơn, phù hợp khi cần gradient direction. Tốc độ nhanh.
- **Laplacian**: Nhạy với nhiễu nhất, edge density cao nhất — nhiều false positive trong ảnh giao thông.

### Phát hiện đường thẳng
- **Standard HoughLines**: Phát hiện được nhiều đường hơn nhưng dễ bị nhiễu (vẽ đường vô hạn qua cả vùng trời).
- **Probabilistic HoughLinesP**: Kiểm soát tốt hơn qua `minLineLength` và `maxLineGap`, kết quả sạch hơn cho làn đường.

**Lựa chọn cho pipeline chính**: Canny + Probabilistic Hough.